# Approach 1 - Simple Regression for per (Peptide, Drug) Pair

## Import Data

In [1]:
import os
from pathlib import Path
from typing import Iterable

os.environ.setdefault("MPLCONFIGDIR", str(Path("data/processed/matplotlib_cache").resolve()))

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CLEAN_MATRIX = Path("data/clean/clean_variant_dyn_log2_min20.csv")
SAMPLE_METADATA = Path("data/clean/clean_sample_metadata.csv")

OUT_DIR = Path("Approach_1_analysis_outputs")


def load_inputs() -> tuple[pd.DataFrame, pd.DataFrame]:
    if not CLEAN_MATRIX.exists():
        raise FileNotFoundError(
            f"Missing {CLEAN_MATRIX}. Run abishai_group_project_try.py first."
        )
    if not SAMPLE_METADATA.exists():
        raise FileNotFoundError(
            f"Missing {SAMPLE_METADATA}. Run abishai_group_project_try.py first."
        )

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    matrix = pd.read_csv(CLEAN_MATRIX, low_memory=False)
    metadata = pd.read_csv(SAMPLE_METADATA)
    metadata = metadata[metadata["measurement_type"] == "dyn"].copy()
    sample_cols = [col for col in matrix.columns if col in set(metadata["sample_column"])]
    metadata = metadata.set_index("sample_column").loc[sample_cols].reset_index()
    return matrix, metadata

matrix, metadata = load_inputs()

In [2]:
matrix.head()

,Variant,Unmod variant,Top canonical protein,Charge,Mass,Variant FDR,Is Decoy,dyn_present_fraction,_dyn_#AEE-788_inBT474 1000nM.Tech replicate 1 of 1,_dyn_#AEE-788_inBT474 100nM.Tech replicate 1 of 1,...,_dyn_#Baricitinib 1000nM.Tech replicate 1 of 1,_dyn_#Baricitinib 100nM.Tech replicate 1 of 1,_dyn_#Baricitinib 10nM.Tech replicate 1 of 1,_dyn_#Baricitinib 30000nM.Tech replicate 1 of 1,_dyn_#Baricitinib 3000nM.Tech replicate 1 of 1,_dyn_#Baricitinib 300nM.Tech replicate 1 of 1,_dyn_#Baricitinib 30nM.Tech replicate 1 of 1,_dyn_#Baricitinib 3nM.Tech replicate 1 of 1,_dyn_#Baricitinib DMSO.Tech replicate 1 of 1,_dyn_#Baricitinib PDPD.Tech replicate 1 of 1
0,.VADPDHDHTGFLTEYVATR.,.VADPDHDHTGFLTEYVATR.,sp|P28482|MK01_HUMAN,2,2144.0,0.0,False,0.990,31.528224,31.676571,...,26.043497,28.681773,28.346634,26.523173,26.326546,27.445770,28.142678,28.023744,27.925130,26.077083
1,.FRHENIIGINDIIR.,.FRHENIIGINDIIR.,sp|P28482|MK01_HUMAN,2,1709.9,0.0,False,0.998,31.898795,32.113125,...,26.125399,27.643683,28.329329,26.431861,26.748680,26.779064,27.735441,27.446401,27.817448,26.542224
2,.ESESTAGSFSLSVR.,.ESESTAGSFSLSVR.,sp|P06239|LCK_HUMAN,2,1456.7,0.0,False,0.860,NaN,NaN,...,23.406654,23.844314,24.122500,23.673575,23.767493,24.196105,23.419055,23.600709,23.613580,22.956172
3,.NYLLSLPHK.,.NYLLSLPHK.,sp|P28482|MK01_HUMAN,2,1084.6,0.0,False,0.994,32.092574,32.258616,...,27.786749,28.842151,29.138876,27.897930,28.165092,28.116295,28.830321,28.435951,25.381031,27.446085
4,.IQDKEGIPPDQQR.,.IQDKEGIPPDQQR.,sp|P62979|RS27A_HUMAN,2,1523.8,0.0,False,0.984,25.290773,25.728348,...,20.730988,23.201829,24.008555,21.256380,21.560949,22.172219,22.919053,23.006211,23.472960,22.475213


In [3]:
# print(list(matrix.columns))

## Simplify Col Names

In [4]:
import re

# These are the non-measurement annotation columns.
# Keep them unchanged.
id_cols = [
    "Variant",
    "Unmod variant",
    "Top canonical protein",
    "Charge",
    "Mass",
    "Variant FDR",
    "Is Decoy",
    "dyn_present_fraction",
]

id_cols = [c for c in id_cols if c in matrix.columns]

# Pattern example:
# _dyn_#AEE-788_inBT474 1000nM.Tech replicate 1 of 1
# _dyn_#AEE-788_inBT474 DMSO.Tech replicate 1 of 1
# _dyn_#AEE-788_inBT474 PDPD.Tech replicate 1 of 1
col_pattern = re.compile(
    r"^_dyn_#(?P<drug>.+?)\s+"
    r"(?P<condition>(?:\d+(?:\.\d+)?)nM|DMSO|PDPD)"
    r"\.Tech replicate (?P<replicate>\d+) of (?P<replicate_total>\d+)$"
)

records = []
rename_dict = {}

for col in matrix.columns:
    if col in id_cols:
        continue

    match = col_pattern.match(col)

    if match is None:
        # Not a dynamic abundance column matching our expected pattern.
        continue

    drug = match.group("drug")
    condition = match.group("condition")
    replicate = int(match.group("replicate"))

    if condition == "DMSO":
        condition_type = "control"
        concentration_nM = 0.0
        simple_col = f"{drug}__DMSO"
    elif condition == "PDPD":
        condition_type = "pdpd"
        concentration_nM = np.nan
        simple_col = f"{drug}__PDPD"
    else:
        condition_type = "treatment"
        concentration_nM = float(condition.replace("nM", ""))
        # Format concentration cleanly: 1000.0 -> 1000nM
        if concentration_nM.is_integer():
            conc_str = f"{int(concentration_nM)}nM"
        else:
            conc_str = f"{concentration_nM:g}nM"
        simple_col = f"{drug}__{conc_str}"

    # If future files contain multiple technical replicates, avoid duplicate names.
    if simple_col in rename_dict.values():
        simple_col = f"{simple_col}__rep{replicate}"

    records.append({
        "original_col": col,
        "simple_col": simple_col,
        "drug": drug,
        "condition": condition,
        "condition_type": condition_type,
        "concentration_nM": concentration_nM,
        "replicate": replicate,
    })

    rename_dict[col] = simple_col

sample_info = pd.DataFrame(records)

# Optional: create a version of matrix with simpler column names.
# This does not modify the original matrix unless you assign it back.
matrix_simple = matrix.rename(columns=rename_dict)

print("Number of parsed dynamic columns:", len(sample_info))
print("Number of ID columns:", len(id_cols))
print("Matrix shape:", matrix.shape)
print("Simplified matrix shape:", matrix_simple.shape)

display(sample_info.head(20))

# Check whether each drug has 1 DMSO control + 8 treatment concentrations.
design_check = (
    sample_info
    .groupby("drug")
    .agg(
        n_control=("condition_type", lambda x: (x == "control").sum()),
        n_treatment=("condition_type", lambda x: (x == "treatment").sum()),
        n_pdpd=("condition_type", lambda x: (x == "pdpd").sum()),
        treatment_concentrations_nM=(
            "concentration_nM",
            lambda x: sorted([v for v in x.dropna().unique() if v > 0])
        ),
    )
    .reset_index()
)

display(design_check.head(20))

# Show possible parsing problems.
bad_design = design_check[
    (design_check["n_control"] != 1) |
    (design_check["n_treatment"] != 8)
]

print("Number of drugs not matching 1 DMSO + 8 concentrations:", len(bad_design))
display(bad_design.head(20))

Number of parsed dynamic columns: 500
Number of ID columns: 8
Matrix shape: (8665, 508)
Simplified matrix shape: (8665, 508)


,original_col,simple_col,drug,condition,condition_type,concentration_nM,replicate
0,_dyn_#AEE-788_inBT474 1000nM.Tech replicate 1 ...,AEE-788_inBT474__1000nM,AEE-788_inBT474,1000nM,treatment,1000.0,1
1,_dyn_#AEE-788_inBT474 100nM.Tech replicate 1 of 1,AEE-788_inBT474__100nM,AEE-788_inBT474,100nM,treatment,100.0,1
2,_dyn_#AEE-788_inBT474 10nM.Tech replicate 1 of 1,AEE-788_inBT474__10nM,AEE-788_inBT474,10nM,treatment,10.0,1
3,_dyn_#AEE-788_inBT474 30000nM.Tech replicate 1...,AEE-788_inBT474__30000nM,AEE-788_inBT474,30000nM,treatment,30000.0,1
4,_dyn_#AEE-788_inBT474 3000nM.Tech replicate 1 ...,AEE-788_inBT474__3000nM,AEE-788_inBT474,3000nM,treatment,3000.0,1
5,_dyn_#AEE-788_inBT474 300nM.Tech replicate 1 of 1,AEE-788_inBT474__300nM,AEE-788_inBT474,300nM,treatment,300.0,1
6,_dyn_#AEE-788_inBT474 30nM.Tech replicate 1 of 1,AEE-788_inBT474__30nM,AEE-788_inBT474,30nM,treatment,30.0,1
7,_dyn_#AEE-788_inBT474 3nM.Tech replicate 1 of 1,AEE-788_inBT474__3nM,AEE-788_inBT474,3nM,treatment,3.0,1
8,_dyn_#AEE-788_inBT474 DMSO.Tech replicate 1 of 1,AEE-788_inBT474__DMSO,AEE-788_inBT474,DMSO,control,0.0,1
9,_dyn_#AEE-788_inBT474 PDPD.Tech replicate 1 of 1,AEE-788_inBT474__PDPD,AEE-788_inBT474,PDPD,pdpd,NaN,1


,drug,n_control,n_treatment,n_pdpd,treatment_concentrations_nM
0,AEE-788_inBT474,1,8,1,"[3.0, 10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0..."
1,AEW-541,1,8,1,"[3.0, 10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0..."
2,AMG-208,1,8,1,"[3.0, 10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0..."
3,AMG-208_withCAKI,1,8,1,"[3.0, 10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0..."
4,AMG-900,1,8,1,"[3.0, 10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0..."
5,ARRY-380,1,8,1,"[3.0, 10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0..."
6,ARRY-380_inBT474,1,8,1,"[3.0, 10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0..."
7,ASP-3026,1,8,1,"[3.0, 10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0..."
8,AT-13148,1,8,1,"[3.0, 10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0..."
9,AT-7519,1,8,1,"[3.0, 10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0..."


Number of drugs not matching 1 DMSO + 8 concentrations: 0


,drug,n_control,n_treatment,n_pdpd,treatment_concentrations_nM


In [5]:
matrix_simple.head()

,Variant,Unmod variant,Top canonical protein,Charge,Mass,Variant FDR,Is Decoy,dyn_present_fraction,AEE-788_inBT474__1000nM,AEE-788_inBT474__100nM,...,Baricitinib__1000nM,Baricitinib__100nM,Baricitinib__10nM,Baricitinib__30000nM,Baricitinib__3000nM,Baricitinib__300nM,Baricitinib__30nM,Baricitinib__3nM,Baricitinib__DMSO,Baricitinib__PDPD
0,.VADPDHDHTGFLTEYVATR.,.VADPDHDHTGFLTEYVATR.,sp|P28482|MK01_HUMAN,2,2144.0,0.0,False,0.990,31.528224,31.676571,...,26.043497,28.681773,28.346634,26.523173,26.326546,27.445770,28.142678,28.023744,27.925130,26.077083
1,.FRHENIIGINDIIR.,.FRHENIIGINDIIR.,sp|P28482|MK01_HUMAN,2,1709.9,0.0,False,0.998,31.898795,32.113125,...,26.125399,27.643683,28.329329,26.431861,26.748680,26.779064,27.735441,27.446401,27.817448,26.542224
2,.ESESTAGSFSLSVR.,.ESESTAGSFSLSVR.,sp|P06239|LCK_HUMAN,2,1456.7,0.0,False,0.860,NaN,NaN,...,23.406654,23.844314,24.122500,23.673575,23.767493,24.196105,23.419055,23.600709,23.613580,22.956172
3,.NYLLSLPHK.,.NYLLSLPHK.,sp|P28482|MK01_HUMAN,2,1084.6,0.0,False,0.994,32.092574,32.258616,...,27.786749,28.842151,29.138876,27.897930,28.165092,28.116295,28.830321,28.435951,25.381031,27.446085
4,.IQDKEGIPPDQQR.,.IQDKEGIPPDQQR.,sp|P62979|RS27A_HUMAN,2,1523.8,0.0,False,0.984,25.290773,25.728348,...,20.730988,23.201829,24.008555,21.256380,21.560949,22.172219,22.919053,23.006211,23.472960,22.475213


In [6]:
matrix_simple.info()

<class 'pandas.DataFrame'>
RangeIndex: 8665 entries, 0 to 8664
Columns: 508 entries, Variant to Baricitinib__PDPD
dtypes: bool(1), float64(503), int64(1), str(3)
memory usage: 33.5 MB


## Step 3 — run one regression per peptide–drug pair

In [7]:
from scipy.stats import linregress

ALPHA = 0.05
MIN_POINTS = 3
MIN_ABS_SLOPE = 0.0   # Keep 0 for now; can add effect-size threshold later.

PEPTIDE_ID_COL = "Variant" if "Variant" in matrix_simple.columns else id_cols[0]

def get_drug_wide_matrix(drug, matrix_simple, sample_info):
    """
    For one drug, extract a wide matrix:
        rows = peptides
        columns = DMSO + treatment concentrations

    Also return the ordered design table for this drug.
    """
    drug_info = sample_info[
        (sample_info["drug"] == drug)
        & (sample_info["condition_type"].isin(["control", "treatment"]))
    ].copy()

    # Exclude columns that somehow are not in matrix_simple.
    drug_info = drug_info[drug_info["simple_col"].isin(matrix_simple.columns)].copy()

    if drug_info.empty:
        return None, None

    nonzero_concs = sorted(
        drug_info.loc[drug_info["concentration_nM"] > 0, "concentration_nM"]
        .dropna()
        .unique()
    )

    if len(nonzero_concs) == 0:
        return None, None

    # Since DMSO has concentration 0, log10(0) is undefined.
    # To include DMSO in the 9-point regression, place it one log-step below
    # the smallest nonzero concentration.
    dmso_model_conc = min(nonzero_concs) / 10.0

    drug_info["model_concentration_nM"] = drug_info["concentration_nM"]
    drug_info.loc[
        drug_info["condition_type"] == "control",
        "model_concentration_nM"
    ] = dmso_model_conc

    drug_info["log10_concentration"] = np.log10(drug_info["model_concentration_nM"])

    # Sort as DMSO first, then increasing concentration.
    drug_info["sort_key"] = np.where(
        drug_info["condition_type"] == "control",
        -1,
        drug_info["concentration_nM"]
    )

    drug_info = drug_info.sort_values("sort_key").reset_index(drop=True)

    drug_cols = drug_info["simple_col"].tolist()

    keep_cols = [PEPTIDE_ID_COL]
    optional_annotation_cols = [
        "Unmod variant",
        "Top canonical protein",
        "Charge",
        "Mass",
        "dyn_present_fraction",
    ]
    keep_cols += [c for c in optional_annotation_cols if c in matrix_simple.columns]

    drug_matrix = matrix_simple[keep_cols + drug_cols].copy()

    return drug_matrix, drug_info


def fit_one_peptide_drug(row, drug, drug_info):
    """
    Fit one linear regression:
        abundance ~ log10_concentration

    One row = one peptide.
    """
    drug_cols = drug_info["simple_col"].tolist()
    x_all = drug_info["log10_concentration"].to_numpy(dtype=float)
    original_conc_all = drug_info["concentration_nM"].to_numpy(dtype=float)

    y_all = pd.to_numeric(row[drug_cols], errors="coerce").to_numpy(dtype=float)

    valid = np.isfinite(x_all) & np.isfinite(y_all)
    x = x_all[valid]
    y = y_all[valid]
    original_conc = original_conc_all[valid]

    n_points = len(y)
    n_missing = len(y_all) - n_points

    base_result = {
        "peptide_id": row[PEPTIDE_ID_COL],
        "drug": drug,
        "n_points": n_points,
        "n_missing": n_missing,
    }

    # Add useful annotations if present.
    for c in ["Unmod variant", "Top canonical protein", "Charge", "Mass", "dyn_present_fraction"]:
        if c in row.index:
            base_result[c] = row[c]

    if n_points < MIN_POINTS or len(np.unique(x)) < 2:
        return {
            **base_result,
            "slope": np.nan,
            "intercept": np.nan,
            "p_value": np.nan,
            "r_value": np.nan,
            "r_squared": np.nan,
            "mse": np.nan,
            "stderr": np.nan,
            "direction": "insufficient_data",
            "significant_raw_p": False,
            "response_label": "insufficient_data",
            "dmso_abundance": np.nan,
            "highest_dose_abundance": np.nan,
            "highest_vs_dmso_delta": np.nan,
        }

    # If y is constant, the slope is effectively zero.
    if np.nanstd(y) == 0:
        slope = 0.0
        intercept = float(y[0])
        p_value = 1.0
        r_value = 0.0
        r_squared = 0.0
        stderr = 0.0
        y_hat = np.repeat(intercept, n_points)
    else:
        reg = linregress(x, y)
        slope = reg.slope
        intercept = reg.intercept
        p_value = reg.pvalue
        r_value = reg.rvalue
        r_squared = reg.rvalue ** 2
        stderr = reg.stderr
        y_hat = intercept + slope * x

    mse = np.mean((y - y_hat) ** 2)

    significant = (
        np.isfinite(p_value)
        and (p_value < ALPHA)
        and (abs(slope) >= MIN_ABS_SLOPE)
    )

    if significant and slope > 0:
        response_label = "increasing_response"
        direction = "increasing"
    elif significant and slope < 0:
        response_label = "decreasing_response"
        direction = "decreasing"
    else:
        response_label = "no_significant_response"
        if slope > 0:
            direction = "positive_not_significant"
        elif slope < 0:
            direction = "negative_not_significant"
        else:
            direction = "flat"

    # Difference between highest available dose and DMSO, when both exist.
    dmso_mask = original_conc == 0
    high_mask = original_conc == np.nanmax(original_conc)

    dmso_abundance = y[dmso_mask][0] if dmso_mask.any() else np.nan
    highest_dose_abundance = y[high_mask][0] if high_mask.any() else np.nan

    if np.isfinite(dmso_abundance) and np.isfinite(highest_dose_abundance):
        highest_vs_dmso_delta = highest_dose_abundance - dmso_abundance
    else:
        highest_vs_dmso_delta = np.nan

    return {
        **base_result,
        "slope": slope,
        "intercept": intercept,
        "p_value": p_value,
        "r_value": r_value,
        "r_squared": r_squared,
        "mse": mse,
        "stderr": stderr,
        "direction": direction,
        "significant_raw_p": significant,
        "response_label": response_label,
        "dmso_abundance": dmso_abundance,
        "highest_dose_abundance": highest_dose_abundance,
        "highest_vs_dmso_delta": highest_vs_dmso_delta,
    }


all_results = []
per_drug_results = {}
per_drug_matrices = {}
per_drug_designs = {}

drugs = sorted(sample_info["drug"].dropna().unique())

print("Number of Drugs to analyze:", len(drugs))
print("Drugs to analyze:", drugs)

from tqdm.auto import tqdm

for drug in tqdm(drugs, desc="Analyzing drugs"):
    drug_matrix, drug_info = get_drug_wide_matrix(
        drug=drug,
        matrix_simple=matrix_simple,
        sample_info=sample_info
    )

    if drug_matrix is None:
        print(f"Skipping {drug}: no valid concentration columns.")
        continue

    per_drug_matrices[drug] = drug_matrix
    per_drug_designs[drug] = drug_info

    print(f"Running {drug}: {drug_matrix.shape[0]} peptides x {len(drug_info)} conditions")

    drug_results = []

    for _, row in drug_matrix.iterrows():
        result = fit_one_peptide_drug(row, drug, drug_info)
        drug_results.append(result)

    drug_results = pd.DataFrame(drug_results)

    # Sort strongest raw-p-value hits first.
    drug_results = drug_results.sort_values(
        by=["p_value", "r_squared", "mse"],
        ascending=[True, False, True],
        na_position="last"
    ).reset_index(drop=True)

    per_drug_results[drug] = drug_results
    all_results.append(drug_results)

regression_results = pd.concat(all_results, ignore_index=True)

print("Finished all drugs.")
print("Total peptide-drug regression results:", regression_results.shape)

display(regression_results.head(20))

Number of Drugs to analyze: 50
Drugs to analyze: ['AEE-788_inBT474', 'AEW-541', 'AMG-208', 'AMG-208_withCAKI', 'AMG-900', 'ARRY-380', 'ARRY-380_inBT474', 'ASP-3026', 'AT-13148', 'AT-7519', 'AT-9283', 'AV-412', 'AV-412_inBT474', 'AXL-1717', 'AZD-1208', 'AZD-1480', 'AZD-2014', 'AZD-4547', 'AZD-5363', 'AZD-5438', 'AZD-6482', 'AZD-7762', 'AZD-8055', 'AZD-8186', 'AZD-8330', 'Abemaciclib', 'Afatinib', 'Afatinib_inBT474', 'Alectinib', 'Alisertib', 'Alvocidib', 'Amuvatinib', 'Apatinib', 'Apitolisib', 'Axitinib', 'BGT-226', 'BI-2536', 'BI-847325', 'BMS-387032', 'BMS-690514', 'BMS-690514_inBT474', 'BMS-754807', 'BMS-777607', 'BMS-777607_withCAKI', 'BMS-911543', 'BYL-719', 'Bafetinib', 'Barasertib', 'Barasertib_HQPA', 'Baricitinib']


Analyzing drugs:   0%|          | 0/50 [00:00<?, ?it/s]

Running AEE-788_inBT474: 8665 peptides x 9 conditions
Running AEW-541: 8665 peptides x 9 conditions
Running AMG-208: 8665 peptides x 9 conditions
Running AMG-208_withCAKI: 8665 peptides x 9 conditions
Running AMG-900: 8665 peptides x 9 conditions
Running ARRY-380: 8665 peptides x 9 conditions
Running ARRY-380_inBT474: 8665 peptides x 9 conditions
Running ASP-3026: 8665 peptides x 9 conditions
Running AT-13148: 8665 peptides x 9 conditions
Running AT-7519: 8665 peptides x 9 conditions
Running AT-9283: 8665 peptides x 9 conditions
Running AV-412: 8665 peptides x 9 conditions
Running AV-412_inBT474: 8665 peptides x 9 conditions
Running AXL-1717: 8665 peptides x 9 conditions
Running AZD-1208: 8665 peptides x 9 conditions
Running AZD-1480: 8665 peptides x 9 conditions
Running AZD-2014: 8665 peptides x 9 conditions
Running AZD-4547: 8665 peptides x 9 conditions
Running AZD-5363: 8665 peptides x 9 conditions
Running AZD-5438: 8665 peptides x 9 conditions
Running AZD-6482: 8665 peptides x 9 co

,peptide_id,drug,n_points,n_missing,Unmod variant,Top canonical protein,Charge,Mass,dyn_present_fraction,slope,...,r_value,r_squared,mse,stderr,direction,significant_raw_p,response_label,dmso_abundance,highest_dose_abundance,highest_vs_dmso_delta
0,.EPLLFSR.,AEE-788_inBT474,9,0,.EPLLFSR.,sp|P53671|LIMK2_HUMAN,2,861.5,0.266,-0.247826,...,-0.949569,0.901682,0.014511,0.030931,decreasing,True,decreasing_response,25.987155,24.694343,-1.292812
1,.VENLLLSNQGTIK.,AEE-788_inBT474,6,3,.VENLLLSNQGTIK.,sp|O14976|GAK_HUMAN,2,1428.8,0.578,-0.179403,...,-0.990576,0.981242,0.001068,0.012402,decreasing,True,decreasing_response,27.150931,26.428798,-0.722133
2,.HSEAATAQREEWK.,AEE-788_inBT474,5,4,.HSEAATAQREEWK.,sp|Q14103|HNRPD_HUMAN,3,1542.7,0.508,0.316937,...,0.996433,0.992879,0.001238,0.015496,increasing,True,increasing_response,24.227953,25.399761,1.171808
3,.ENGGASHPLLDQR.,AEE-788_inBT474,9,0,.ENGGASHPLLDQR.,sp|P54760|EPHB4_HUMAN,2,1393.7,0.742,-0.211815,...,-0.930829,0.866443,0.014985,0.031432,decreasing,True,decreasing_response,28.914391,27.747912,-1.166479
4,.GDSFTHTPPLDPQELDILK.,AEE-788_inBT474,9,0,.GDSFTHTPPLDPQELDILK.,sp|P00533|EGFR_HUMAN,3,2123.0,0.478,-0.454618,...,-0.929797,0.864522,0.070178,0.068021,decreasing,True,decreasing_response,26.702266,24.879048,-1.823218
5,.SEEEKDQEKQQMFENK.,AEE-788_inBT474,9,0,.SEEEKDQEKQQMFENK.,sp|Q9H2G2|SLK_HUMAN,3,2026.9,0.260,-0.293050,...,-0.926823,0.859002,0.030544,0.044875,decreasing,True,decreasing_response,25.100050,23.532552,-1.567498
6,.IPLENLQIIR.,AEE-788_inBT474,7,2,.IPLENLQIIR.,sp|P00533|EGFR_HUMAN,2,1208.7,0.810,-0.744918,...,-0.962605,0.926608,0.043957,0.093756,decreasing,True,decreasing_response,NaN,24.973600,NaN
7,.FAGHSEAGGGSGDR.,AEE-788_inBT474,9,0,.FAGHSEAGGGSGDR.,sp|O96013|PAK4_HUMAN,3,1304.5,0.298,-0.290609,...,-0.913830,0.835085,0.036138,0.048812,decreasing,True,decreasing_response,31.067021,29.957123,-1.109898
8,.NYVENRPK.,AEE-788_inBT474,8,1,.NYVENRPK.,sp|P45984|MK09_HUMAN,2,1019.5,0.220,-0.231899,...,-0.936122,0.876325,0.017466,0.035566,decreasing,True,decreasing_response,29.671029,28.698561,-0.972467
9,.LATGEEEGGGSSSK.,AEE-788_inBT474,9,0,.LATGEEEGGGSSSK.,sp|P00519|ABL1_HUMAN,2,1308.6,0.200,-0.441368,...,-0.905653,0.820207,0.092527,0.078105,decreasing,True,decreasing_response,27.438808,24.958634,-2.480174


## Step 4 -- Analysis

In [8]:
drug_summary = (
    regression_results
    .groupby("drug")
    .agg(
        n_tests=("peptide_id", "count"),
        n_significant_raw_p=("significant_raw_p", "sum"),
        n_increasing=("response_label", lambda x: (x == "increasing_response").sum()),
        n_decreasing=("response_label", lambda x: (x == "decreasing_response").sum()),
        median_abs_slope=("slope", lambda x: np.nanmedian(np.abs(x))),
        median_r_squared=("r_squared", "median"),
        median_mse=("mse", "median"),
    )
    .reset_index()
)

drug_summary["significant_raw_p_fraction"] = (
    drug_summary["n_significant_raw_p"] / drug_summary["n_tests"]
)

drug_summary = drug_summary.sort_values(
    by="n_significant_raw_p",
    ascending=False
).reset_index(drop=True)

display(drug_summary.head(30))

,drug,n_tests,n_significant_raw_p,n_increasing,n_decreasing,median_abs_slope,median_r_squared,median_mse,significant_raw_p_fraction
0,Baricitinib,8665,2511,21,2490,0.361420,0.514863,0.238786,0.289786
1,Amuvatinib,8665,1561,60,1501,0.160981,0.344402,0.094623,0.180150
2,AMG-208,8665,1348,16,1332,0.211168,0.337503,0.175012,0.155568
3,BI-2536,8665,1270,6,1264,0.282013,0.442038,0.224362,0.146567
4,BMS-777607_withCAKI,8665,1025,28,997,0.158412,0.312079,0.101668,0.118292
5,AZD-6482,8665,1018,971,47,0.108173,0.206782,0.084144,0.117484
6,AEW-541,8665,983,10,973,0.195550,0.272901,0.215977,0.113445
7,Alisertib,8665,865,86,779,0.137993,0.249217,0.114658,0.099827
8,BMS-690514_inBT474,8665,826,31,795,0.125766,0.174184,0.143497,0.095326
9,AZD-5363,8665,789,717,72,0.173164,0.259471,0.166957,0.091056


In [9]:
# Strongest peptide-drug hits using raw p-value only

top_hits_overall = (
    regression_results
    .dropna(subset=["p_value"])
    .sort_values(
        by=["p_value", "r_squared", "mse"],
        ascending=[True, False, True]
    )
    .reset_index(drop=True)
)

display(top_hits_overall.head(50))


# Top significant hits per drug
top_hits_per_drug = (
    regression_results
    .dropna(subset=["p_value"])
    .sort_values(
        by=["drug", "p_value", "r_squared", "mse"],
        ascending=[True, True, False, True]
    )
    .groupby("drug")
    .head(10)
    .reset_index(drop=True)
)

display(top_hits_per_drug)

,peptide_id,drug,n_points,n_missing,Unmod variant,Top canonical protein,Charge,Mass,dyn_present_fraction,slope,...,r_value,r_squared,mse,stderr,direction,significant_raw_p,response_label,dmso_abundance,highest_dose_abundance,highest_vs_dmso_delta
0,.DDFTEFGK.,AT-7519,9,0,.DDFTEFGK.,sp|O75822|EIF3J_HUMAN,2,958.4,0.744,-0.526005,...,-0.986659,0.973495,1.632249e-02,0.032805,decreasing,True,decreasing_response,22.662926,20.298829,-2.364097
1,.LSKEDIER.,BMS-777607_withCAKI,9,0,.LSKEDIER.,sp|P11142|HSP7C_HUMAN,2,989.5,0.800,-0.219440,...,-0.985462,0.971135,3.101243e-03,0.014299,decreasing,True,decreasing_response,24.784816,23.700282,-1.084533
2,.LATGEEEGGGSSSK.,ARRY-380_inBT474,9,0,.LATGEEEGGGSSSK.,sp|P00519|ABL1_HUMAN,2,1308.6,0.200,-0.359645,...,-0.975916,0.952412,1.400355e-02,0.030385,decreasing,True,decreasing_response,27.393212,25.818606,-1.574605
3,.VSTAVLSITAK.,Amuvatinib,8,1,.VSTAVLSITAK.,sp|Q99460|PSMD1_HUMAN,2,1089.6,0.264,-0.318340,...,-0.985700,0.971604,4.600728e-03,0.022218,decreasing,True,decreasing_response,NaN,19.751222,NaN
4,.LQPQEISPPPTANLDR.,BMS-754807,9,0,.LQPQEISPPPTANLDR.,sp|Q05397|FAK1_HUMAN,2,1775.9,0.990,-1.238528,...,-0.975127,0.950872,1.717260e-01,0.106405,decreasing,True,decreasing_response,25.033261,19.706865,-5.326395
5,.SNPEDQILYQTER.,Abemaciclib,8,1,.SNPEDQILYQTER.,sp|Q14165|MLEC_HUMAN,2,1592.7,0.634,0.234480,...,0.985287,0.970791,3.976085e-03,0.016605,increasing,True,increasing_response,19.961553,21.263618,1.302065
6,.-17.027QVHPDTGISSK.,Afatinib_inBT474,9,0,.QVHPDTGISSK.,sp|P58876|H2B1D_HUMAN,2,1151.6,0.222,0.284802,...,0.974980,0.950586,9.136142e-03,0.024543,increasing,True,increasing_response,29.444882,30.866365,1.421484
7,.NNASTDYDLSDK.,AT-7519,9,0,.NNASTDYDLSDK.,sp|P39023|RL3_HUMAN,2,1342.6,0.790,-0.590700,...,-0.974597,0.949839,3.992660e-02,0.051307,decreasing,True,decreasing_response,24.310566,21.432473,-2.878093
8,.WIHQLK.,AEW-541,8,1,.WIHQLK.,sp|Q06187|BTK_HUMAN,2,824.5,0.800,-0.350791,...,-0.981702,0.963738,1.111866e-02,0.027779,decreasing,True,decreasing_response,25.295055,23.452619,-1.842436
9,.QQTQSALEQR.,AZD-1480,9,0,.QQTQSALEQR.,sp|Q86YS7|C2CD5_HUMAN,2,1188.6,0.780,0.166848,...,0.969469,0.939870,3.859070e-03,0.015951,increasing,True,increasing_response,23.851957,24.638486,0.786529


,peptide_id,drug,n_points,n_missing,Unmod variant,Top canonical protein,Charge,Mass,dyn_present_fraction,slope,...,r_value,r_squared,mse,stderr,direction,significant_raw_p,response_label,dmso_abundance,highest_dose_abundance,highest_vs_dmso_delta
0,.EPLLFSR.,AEE-788_inBT474,9,0,.EPLLFSR.,sp|P53671|LIMK2_HUMAN,2,861.5,0.266,-0.247826,...,-0.949569,0.901682,1.451081e-02,0.030931,decreasing,True,decreasing_response,25.987155,24.694343,-1.292812
1,.VENLLLSNQGTIK.,AEE-788_inBT474,6,3,.VENLLLSNQGTIK.,sp|O14976|GAK_HUMAN,2,1428.8,0.578,-0.179403,...,-0.990576,0.981242,1.067522e-03,0.012402,decreasing,True,decreasing_response,27.150931,26.428798,-0.722133
2,.HSEAATAQREEWK.,AEE-788_inBT474,5,4,.HSEAATAQREEWK.,sp|Q14103|HNRPD_HUMAN,3,1542.7,0.508,0.316937,...,0.996433,0.992879,1.237982e-03,0.015496,increasing,True,increasing_response,24.227953,25.399761,1.171808
3,.ENGGASHPLLDQR.,AEE-788_inBT474,9,0,.ENGGASHPLLDQR.,sp|P54760|EPHB4_HUMAN,2,1393.7,0.742,-0.211815,...,-0.930829,0.866443,1.498494e-02,0.031432,decreasing,True,decreasing_response,28.914391,27.747912,-1.166479
4,.GDSFTHTPPLDPQELDILK.,AEE-788_inBT474,9,0,.GDSFTHTPPLDPQELDILK.,sp|P00533|EGFR_HUMAN,3,2123.0,0.478,-0.454618,...,-0.929797,0.864522,7.017771e-02,0.068021,decreasing,True,decreasing_response,26.702266,24.879048,-1.823218
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,.ATVLESEGTRESAINVAEGKK.,Baricitinib,3,6,.ATVLESEGTRESAINVAEGKK.,sp|Q9UJZ1|STML2_HUMAN,3,2189.1,0.328,-0.380090,...,-1.000000,1.000000,5.893900e-08,0.000195,decreasing,True,decreasing_response,22.150815,21.010657,-1.140158
496,.EVDIGIPDATGRLEILQIHTK.,Baricitinib,6,3,.EVDIGIPDATGRLEILQIHTK.,sp|P55072|TERA_HUMAN,3,2318.3,0.352,-0.606955,...,-0.984266,0.968779,3.594707e-02,0.054480,decreasing,True,decreasing_response,22.186375,19.174021,-3.012353
497,.YLKDVTLQK.,Baricitinib,9,0,.YLKDVTLQK.,sp|P18621|RL17_HUMAN,3,1107.6,0.892,-0.705943,...,-0.922495,0.850997,1.890702e-01,0.111649,decreasing,True,decreasing_response,24.517553,21.543261,-2.974292
498,.AATFGLILDDVSLTHLTFGK.,Baricitinib,9,0,.AATFGLILDDVSLTHLTFGK.,sp|P35232|PHB_HUMAN,2,2119.1,0.862,0.417214,...,0.922477,0.850964,6.605608e-02,0.065993,increasing,True,increasing_response,26.147519,27.895505,1.747986


In [10]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

regression_results.to_csv(OUT_DIR / "step3_all_peptide_drug_linear_regression_raw_p.csv", index=False)
drug_summary.to_csv(OUT_DIR / "step4_drug_level_summary_raw_p.csv", index=False)
top_hits_overall.to_csv(OUT_DIR / "step4_top_hits_overall_raw_p.csv", index=False)
top_hits_per_drug.to_csv(OUT_DIR / "step4_top_hits_per_drug_raw_p.csv", index=False)

print("Saved outputs to:", OUT_DIR.resolve())

Saved outputs to: C:\Users\司宸\Desktop\Spring_2026\CSE291B\CSE-291B-Project\Approach_1_analysis_outputs


In [11]:
# Report-ready summary statistics

ALPHA = 0.05

total_pairs = len(regression_results)
total_drugs = regression_results["drug"].nunique()
total_peptides = regression_results["peptide_id"].nunique()

expected_values = regression_results["n_points"].sum() + regression_results["n_missing"].sum()
observed_values = regression_results["n_points"].sum()
missing_values = regression_results["n_missing"].sum()

valid_regression_mask = regression_results["p_value"].notna()
invalid_regression_mask = ~valid_regression_mask

valid_pairs = valid_regression_mask.sum()
invalid_pairs = invalid_regression_mask.sum()

sig_005 = (regression_results["p_value"] <= 0.05).sum()
sig_001 = (regression_results["p_value"] <= 0.01).sum()
sig_0001 = (regression_results["p_value"] <= 0.001).sum()

increasing = (regression_results["response_label"] == "increasing_response").sum()
decreasing = (regression_results["response_label"] == "decreasing_response").sum()
no_response = (regression_results["response_label"] == "no_significant_response").sum()

report_summary = pd.DataFrame({
    "metric": [
        "number_of_peptides",
        "number_of_drugs",
        "total_peptide_drug_pairs",
        "expected_abundance_values",
        "observed_abundance_values",
        "missing_abundance_values",
        "missing_abundance_fraction",
        "valid_regression_pairs",
        "invalid_regression_pairs",
        "valid_regression_fraction",
        "raw_p_le_0.05",
        "raw_p_le_0.05_fraction",
        "raw_p_le_0.01",
        "raw_p_le_0.01_fraction",
        "raw_p_le_0.001",
        "raw_p_le_0.001_fraction",
        "increasing_raw_significant",
        "decreasing_raw_significant",
        "no_significant_response",
        "median_slope",
        "median_abs_slope",
        "median_r_squared",
        "median_mse",
    ],
    "value": [
        total_peptides,
        total_drugs,
        total_pairs,
        expected_values,
        observed_values,
        missing_values,
        missing_values / expected_values,
        valid_pairs,
        invalid_pairs,
        valid_pairs / total_pairs,
        sig_005,
        sig_005 / valid_pairs,
        sig_001,
        sig_001 / valid_pairs,
        sig_0001,
        sig_0001 / valid_pairs,
        increasing,
        decreasing,
        no_response,
        regression_results["slope"].median(skipna=True),
        regression_results["slope"].abs().median(skipna=True),
        regression_results["r_squared"].median(skipna=True),
        regression_results["mse"].median(skipna=True),
    ]
})

display(report_summary)

,metric,value
0,number_of_peptides,8.665000e+03
1,number_of_drugs,5.000000e+01
2,total_peptide_drug_pairs,4.332500e+05
3,expected_abundance_values,3.899250e+06
4,observed_abundance_values,1.923306e+06
5,missing_abundance_values,1.975944e+06
6,missing_abundance_fraction,5.067498e-01
7,valid_regression_pairs,2.536490e+05
8,invalid_regression_pairs,1.796010e+05
9,valid_regression_fraction,5.854564e-01


In [12]:
# Optional stricter rule:
# Keep only peptide-drug pairs with at most 1 missing abundance value.

MAX_MISSING_ALLOWED = 1

strict_results = regression_results[
    regression_results["p_value"].notna()
    & (regression_results["n_missing"] <= MAX_MISSING_ALLOWED)
].copy()

strict_summary = pd.DataFrame({
    "metric": [
        "strict_valid_pairs",
        "strict_invalid_or_excluded_pairs",
        "strict_valid_fraction",
        "strict_raw_p_le_0.05",
        "strict_raw_p_le_0.05_fraction",
        "strict_increasing_raw_significant",
        "strict_decreasing_raw_significant",
        "strict_median_abs_slope",
        "strict_median_r_squared",
        "strict_median_mse",
    ],
    "value": [
        len(strict_results),
        len(regression_results) - len(strict_results),
        len(strict_results) / len(regression_results),
        (strict_results["p_value"] <= 0.05).sum(),
        (strict_results["p_value"] <= 0.05).mean(),
        (strict_results["response_label"] == "increasing_response").sum(),
        (strict_results["response_label"] == "decreasing_response").sum(),
        strict_results["slope"].abs().median(skipna=True),
        strict_results["r_squared"].median(skipna=True),
        strict_results["mse"].median(skipna=True),
    ]
})

display(strict_summary)

,metric,value
0,strict_valid_pairs,155026.000000
1,strict_invalid_or_excluded_pairs,278224.000000
2,strict_valid_fraction,0.357821
3,strict_raw_p_le_0.05,20732.000000
4,strict_raw_p_le_0.05_fraction,0.133732
5,strict_increasing_raw_significant,7294.000000
6,strict_decreasing_raw_significant,13438.000000
7,strict_median_abs_slope,0.109040
8,strict_median_r_squared,0.136972
9,strict_median_mse,0.156644


In [13]:
# Strict missingness filter for final report:
# More than 20% missing out of 9 points means n_missing >= 2.
# Therefore, valid pairs must satisfy n_missing <= 1.

MAX_MISSING_ALLOWED = 1

strict_results = regression_results[
    regression_results["p_value"].notna()
    & (regression_results["n_missing"] <= MAX_MISSING_ALLOWED)
].copy()

strict_invalid_pairs = len(regression_results) - len(strict_results)

strict_summary = pd.DataFrame({
    "metric": [
        "strict_valid_pairs",
        "strict_invalid_pairs",
        "strict_valid_fraction",
        "strict_raw_p_le_0.05",
        "strict_raw_p_le_0.05_fraction",
        "strict_raw_p_le_0.01",
        "strict_raw_p_le_0.01_fraction",
        "strict_raw_p_le_0.001",
        "strict_raw_p_le_0.001_fraction",
        "strict_increasing_raw_significant",
        "strict_decreasing_raw_significant",
        "strict_median_slope",
        "strict_median_abs_slope",
        "strict_median_r_squared",
        "strict_median_mse",
    ],
    "value": [
        len(strict_results),
        strict_invalid_pairs,
        len(strict_results) / len(regression_results),
        (strict_results["p_value"] <= 0.05).sum(),
        (strict_results["p_value"] <= 0.05).mean(),
        (strict_results["p_value"] <= 0.01).sum(),
        (strict_results["p_value"] <= 0.01).mean(),
        (strict_results["p_value"] <= 0.001).sum(),
        (strict_results["p_value"] <= 0.001).mean(),
        (strict_results["response_label"] == "increasing_response").sum(),
        (strict_results["response_label"] == "decreasing_response").sum(),
        strict_results["slope"].median(skipna=True),
        strict_results["slope"].abs().median(skipna=True),
        strict_results["r_squared"].median(skipna=True),
        strict_results["mse"].median(skipna=True),
    ]
})

display(strict_summary)

strict_summary.to_csv(
    OUT_DIR / "step4_strict_missingness_summary_raw_p.csv",
    index=False
)

,metric,value
0,strict_valid_pairs,155026.000000
1,strict_invalid_pairs,278224.000000
2,strict_valid_fraction,0.357821
3,strict_raw_p_le_0.05,20732.000000
4,strict_raw_p_le_0.05_fraction,0.133732
5,strict_raw_p_le_0.01,5693.000000
6,strict_raw_p_le_0.01_fraction,0.036723
7,strict_raw_p_le_0.001,752.000000
8,strict_raw_p_le_0.001_fraction,0.004851
9,strict_increasing_raw_significant,7294.000000


In [14]:
# Report figures for Approach 1

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

FIG_DIR = OUT_DIR / "report_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Choose which result table to plot.
# For the final report, use strict_results.
plot_results = strict_results.copy()

print("Number of pairs used for plotting:", len(plot_results))

Number of pairs used for plotting: 155026


In [15]:
# Figure 1: Missing values per peptide-drug pair

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(regression_results["n_missing"].dropna(), bins=np.arange(-0.5, 10.5, 1))
ax.set_xlabel("Number of missing values out of 9 conditions")
ax.set_ylabel("Number of peptide-drug pairs")
ax.set_title("Missingness per peptide-drug pair")
fig.tight_layout()

fig.savefig(FIG_DIR / "fig_missing_values_per_pair.png", dpi=300)
display(fig)
plt.close(fig)

<Figure size 700x400 with 1 Axes>

In [16]:
# Figure 2: Raw p-value histogram

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(plot_results["p_value"].dropna(), bins=50)
ax.set_xlabel("Raw p-value")
ax.set_ylabel("Number of peptide-drug pairs")
ax.set_title("Distribution of raw regression p-values")
fig.tight_layout()

fig.savefig(FIG_DIR / "fig_raw_pvalue_histogram.png", dpi=300)
display(fig)
plt.close(fig)

<Figure size 700x400 with 1 Axes>

In [17]:
# Figure 3: R-squared histogram

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(plot_results["r_squared"].dropna(), bins=50)
ax.set_xlabel("$R^2$")
ax.set_ylabel("Number of peptide-drug pairs")
ax.set_title("Distribution of regression $R^2$")
fig.tight_layout()

fig.savefig(FIG_DIR / "fig_r_squared_histogram.png", dpi=300)
display(fig)
plt.close(fig)

<Figure size 700x400 with 1 Axes>

In [18]:
# Figure 4: MSE histogram, clipped at the 99th percentile for readability

mse_values = plot_results["mse"].dropna()
mse_clip = mse_values.quantile(0.99)
mse_values_clipped = mse_values[mse_values <= mse_clip]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(mse_values_clipped, bins=50)
ax.set_xlabel("Mean squared error")
ax.set_ylabel("Number of peptide-drug pairs")
ax.set_title("Distribution of regression MSE")
fig.tight_layout()

fig.savefig(FIG_DIR / "fig_mse_histogram_clipped_99pct.png", dpi=300)
display(fig)
plt.close(fig)

<Figure size 700x400 with 1 Axes>

In [19]:
# Figure 5: Top drugs by raw significant count

drug_summary_strict = (
    plot_results
    .groupby("drug")
    .agg(
        n_tests=("peptide_id", "count"),
        n_significant_raw_p=("p_value", lambda x: (x <= 0.05).sum()),
        n_increasing=("response_label", lambda x: (x == "increasing_response").sum()),
        n_decreasing=("response_label", lambda x: (x == "decreasing_response").sum()),
        median_abs_slope=("slope", lambda x: np.nanmedian(np.abs(x))),
        median_r_squared=("r_squared", "median"),
        median_mse=("mse", "median"),
    )
    .reset_index()
)

drug_summary_strict["significant_raw_p_fraction"] = (
    drug_summary_strict["n_significant_raw_p"] / drug_summary_strict["n_tests"]
)

drug_summary_strict = drug_summary_strict.sort_values(
    "n_significant_raw_p",
    ascending=False
).reset_index(drop=True)

top_drugs = drug_summary_strict.head(15)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(top_drugs["drug"], top_drugs["n_significant_raw_p"])
ax.set_xlabel("Drug")
ax.set_ylabel("Number of raw p <= 0.05 peptide responses")
ax.set_title("Top drugs by raw-significant peptide count")
ax.tick_params(axis="x", rotation=75)
fig.tight_layout()

fig.savefig(FIG_DIR / "fig_top_drugs_raw_significant_count.png", dpi=300)
display(fig)
plt.close(fig)

display(drug_summary_strict.head(20))

<Figure size 900x500 with 1 Axes>

,drug,n_tests,n_significant_raw_p,n_increasing,n_decreasing,median_abs_slope,median_r_squared,median_mse,significant_raw_p_fraction
0,Baricitinib,3067,2064,4,2060,0.375527,0.526396,0.274886,0.672970
1,Amuvatinib,3845,1261,27,1234,0.151464,0.321090,0.103752,0.327958
2,AMG-208,3866,1100,3,1097,0.214482,0.332628,0.191544,0.284532
3,BMS-777607_withCAKI,2815,849,9,840,0.159837,0.312030,0.113290,0.301599
4,BI-2536,2382,826,2,824,0.278105,0.429507,0.244804,0.346767
5,AZD-6482,3869,801,794,7,0.102225,0.186777,0.092266,0.207030
6,AEW-541,3305,782,3,779,0.192886,0.277259,0.229973,0.236611
7,Alisertib,3860,685,40,645,0.134902,0.237235,0.126257,0.177461
8,BMS-690514_inBT474,3580,634,15,619,0.112163,0.141097,0.156370,0.177095
9,AZD-8055,3531,607,596,11,0.138071,0.235280,0.125127,0.171906


In [20]:
# Figure 6: Slope vs -log10(raw p-value)

scatter_df = plot_results.dropna(subset=["slope", "p_value"]).copy()
scatter_df = scatter_df[scatter_df["p_value"] > 0]

# Optional downsampling for faster plotting if there are many points.
MAX_POINTS = 100000
if len(scatter_df) > MAX_POINTS:
    scatter_df = scatter_df.sample(MAX_POINTS, random_state=0)

scatter_df["neg_log10_p"] = -np.log10(scatter_df["p_value"])

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(scatter_df["slope"], scatter_df["neg_log10_p"], s=4, alpha=0.4)
ax.axhline(-np.log10(0.05), linestyle="--", linewidth=1)
ax.set_xlabel("Regression slope")
ax.set_ylabel("-log10(raw p-value)")
ax.set_title("Slope vs. raw significance")
fig.tight_layout()

fig.savefig(FIG_DIR / "fig_slope_vs_neglog10_pvalue.png", dpi=300)
display(fig)
plt.close(fig)

<Figure size 700x500 with 1 Axes>

## Cells to pick example pairs and plot dose-response curves

In [21]:
# Select high-confidence and likely no-response examples

high_confidence_examples = (
    plot_results
    .dropna(subset=["p_value", "r_squared", "slope"])
    .query("p_value <= 0.001 and r_squared >= 0.80")
    .sort_values(["p_value", "r_squared"], ascending=[True, False])
    .head(10)
)

no_response_examples = (
    plot_results
    .dropna(subset=["p_value", "r_squared", "slope"])
    .query("p_value >= 0.80 and r_squared <= 0.05 and abs(slope) <= 0.05")
    .sort_values(["r_squared", "p_value"], ascending=[True, False])
    .head(10)
)

display(high_confidence_examples[
    ["peptide_id", "drug", "n_points", "n_missing", "slope", "p_value", "r_squared", "mse", "response_label"]
])

display(no_response_examples[
    ["peptide_id", "drug", "n_points", "n_missing", "slope", "p_value", "r_squared", "mse", "response_label"]
])

,peptide_id,drug,n_points,n_missing,slope,p_value,r_squared,mse,response_label
77985,.DDFTEFGK.,AT-7519,9,0,-0.526005,8.914486e-07,0.973495,0.016322,decreasing_response
372595,.LSKEDIER.,BMS-777607_withCAKI,9,0,-0.219440,1.202682e-06,0.971135,0.003101,decreasing_response
51990,.LATGEEEGGGSSSK.,ARRY-380_inBT474,9,0,-0.359645,6.972380e-06,0.952412,0.014004,decreasing_response
268615,.VSTAVLSITAK.,Amuvatinib,8,1,-0.318340,7.233006e-06,0.971604,0.004601,decreasing_response
355265,.LQPQEISPPPTANLDR.,BMS-754807,9,0,-1.238528,7.799434e-06,0.950872,0.171726,decreasing_response
216625,.SNPEDQILYQTER.,Abemaciclib,8,1,0.234480,7.874493e-06,0.970791,0.003976,increasing_response
233955,.-17.027QVHPDTGISSK.,Afatinib_inBT474,9,0,0.284802,7.960552e-06,0.950586,0.009136,increasing_response
77986,.NNASTDYDLSDK.,AT-7519,9,0,-0.590700,8.391864e-06,0.949839,0.039927,decreasing_response
8665,.WIHQLK.,AEW-541,8,1,-0.350791,1.510764e-05,0.963738,0.011119,decreasing_response
129975,.QQTQSALEQR.,AZD-1480,9,0,0.166848,1.589174e-05,0.939870,0.003859,increasing_response


,peptide_id,drug,n_points,n_missing,slope,p_value,r_squared,mse,response_label
5083,.EGQFKDIITK.,AEE-788_inBT474,9,0,-8.018662e-07,0.999991,1.944668e-11,0.071643,no_significant_response
92177,.IQDKEGIPPDQQR.,AT-9283,9,0,-7.813576e-06,0.999969,2.252679e-10,0.587240,no_significant_response
187024,.LSDGVAVLK.,AZD-7762,9,0,4.227624e-06,0.999969,2.319053e-10,0.166993,no_significant_response
237804,.EVETHANNSSIELEK.,Afatinib_inBT474,9,0,5.729792e-06,0.999962,3.478049e-10,0.204530,no_significant_response
394657,.FIIPQIVK.,BYL-719,9,0,-1.033506e-05,0.999955,4.902192e-10,0.472119,no_significant_response
308155,.SHDFYSHELSSPVDSPSSLR.,BGT-226,9,0,-4.092924e-06,0.999952,5.451465e-10,0.066584,no_significant_response
30808,.YQILPLHSQIPR.,AMG-208_withCAKI,8,1,-8.401725e-06,0.999955,5.651964e-10,0.265327,no_significant_response
204218,.TPLHEIALSIK.,AZD-8186,9,0,-5.727169e-06,0.999943,7.697686e-10,0.092328,no_significant_response
334636,.TYLLDFR.,BMS-387032,9,0,-8.545963e-06,0.999933,1.095312e-09,0.144477,no_significant_response
264779,.QASIQHIQNAIDTEK.,Alvocidib,9,0,-7.361499e-06,0.999931,1.149250e-09,0.102172,no_significant_response


In [22]:
# Generate LaTeX tables for selected examples

print("High-confidence examples LaTeX:")
print(
    high_confidence_examples[
        ["peptide_id", "drug", "n_points", "n_missing", "slope", "p_value", "r_squared", "mse", "response_label"]
    ]
    .head(5)
    .to_latex(index=False, float_format="%.4g")
)

print("\nLikely no-response examples LaTeX:")
print(
    no_response_examples[
        ["peptide_id", "drug", "n_points", "n_missing", "slope", "p_value", "r_squared", "mse", "response_label"]
    ]
    .head(5)
    .to_latex(index=False, float_format="%.4g")
)

High-confidence examples LaTeX:
\begin{tabular}{llrrrrrrl}
\toprule
peptide_id & drug & n_points & n_missing & slope & p_value & r_squared & mse & response_label \\
\midrule
.DDFTEFGK. & AT-7519 & 9 & 0 & -0.526 & 8.914e-07 & 0.9735 & 0.01632 & decreasing_response \\
.LSKEDIER. & BMS-777607_withCAKI & 9 & 0 & -0.2194 & 1.203e-06 & 0.9711 & 0.003101 & decreasing_response \\
.LATGEEEGGGSSSK. & ARRY-380_inBT474 & 9 & 0 & -0.3596 & 6.972e-06 & 0.9524 & 0.014 & decreasing_response \\
.VSTAVLSITAK. & Amuvatinib & 8 & 1 & -0.3183 & 7.233e-06 & 0.9716 & 0.004601 & decreasing_response \\
.LQPQEISPPPTANLDR. & BMS-754807 & 9 & 0 & -1.239 & 7.799e-06 & 0.9509 & 0.1717 & decreasing_response \\
\bottomrule
\end{tabular}


Likely no-response examples LaTeX:
\begin{tabular}{llrrrrrrl}
\toprule
peptide_id & drug & n_points & n_missing & slope & p_value & r_squared & mse & response_label \\
\midrule
.EGQFKDIITK. & AEE-788_inBT474 & 9 & 0 & -8.019e-07 & 1 & 1.945e-11 & 0.07164 & no_significant_response \

In [23]:
# Dose-response plot for one peptide-drug pair

import re

def safe_filename(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))

def plot_peptide_drug_response(peptide_id, drug):
    drug_matrix = per_drug_matrices[drug]
    drug_info = per_drug_designs[drug].copy()

    row = drug_matrix.loc[drug_matrix[PEPTIDE_ID_COL] == peptide_id]
    if row.empty:
        raise ValueError(f"Peptide {peptide_id} not found for drug {drug}.")

    row = row.iloc[0]
    drug_cols = drug_info["simple_col"].tolist()

    y = pd.to_numeric(row[drug_cols], errors="coerce").to_numpy(dtype=float)
    x = drug_info["log10_concentration"].to_numpy(dtype=float)
    original_conc = drug_info["concentration_nM"].to_numpy(dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)

    result_row = regression_results[
        (regression_results["peptide_id"] == peptide_id)
        & (regression_results["drug"] == drug)
    ].iloc[0]

    slope = result_row["slope"]
    intercept = result_row["intercept"]
    p_value = result_row["p_value"]
    r_squared = result_row["r_squared"]

    x_line = np.linspace(np.nanmin(x[valid]), np.nanmax(x[valid]), 100)
    y_line = intercept + slope * x_line

    labels = [
        "DMSO" if c == 0 else f"{c:g} nM"
        for c in original_conc
    ]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(x[valid], y[valid], s=40)
    ax.plot(x_line, y_line)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_xlabel("Condition")
    ax.set_ylabel("log2 peptide abundance")
    ax.set_title(
        f"{peptide_id} under {drug}\n"
        f"slope={slope:.3g}, p={p_value:.3g}, R²={r_squared:.3g}"
    )
    fig.tight_layout()

    out_path = FIG_DIR / f"dose_response_{safe_filename(peptide_id)}_{safe_filename(drug)}.png"
    fig.savefig(out_path, dpi=300)
    display(fig)
    plt.close(fig)

    return out_path

In [24]:
# Plot several high-confidence examples

for _, example in high_confidence_examples.head(3).iterrows():
    path = plot_peptide_drug_response(example["peptide_id"], example["drug"])
    print("Saved:", path)

<Figure size 700x400 with 1 Axes>

Saved: Approach_1_analysis_outputs\report_figures\dose_response_.DDFTEFGK._AT-7519.png


<Figure size 700x400 with 1 Axes>

Saved: Approach_1_analysis_outputs\report_figures\dose_response_.LSKEDIER._BMS-777607_withCAKI.png


<Figure size 700x400 with 1 Axes>

Saved: Approach_1_analysis_outputs\report_figures\dose_response_.LATGEEEGGGSSSK._ARRY-380_inBT474.png


In [25]:
# Plot several likely no-response examples

for _, example in no_response_examples.head(3).iterrows():
    path = plot_peptide_drug_response(example["peptide_id"], example["drug"])
    print("Saved:", path)

<Figure size 700x400 with 1 Axes>

Saved: Approach_1_analysis_outputs\report_figures\dose_response_.EGQFKDIITK._AEE-788_inBT474.png


<Figure size 700x400 with 1 Axes>

Saved: Approach_1_analysis_outputs\report_figures\dose_response_.IQDKEGIPPDQQR._AT-9283.png


<Figure size 700x400 with 1 Axes>

Saved: Approach_1_analysis_outputs\report_figures\dose_response_.LSDGVAVLK._AZD-7762.png


In [26]:
# Combined 3-by-2 dose-response figure:
# Top row: high-confidence response examples
# Bottom row: likely no-response examples

example_grid = pd.concat([
    high_confidence_examples.head(3).assign(example_type="High-confidence response"),
    no_response_examples.head(3).assign(example_type="Likely no response")
], ignore_index=True)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, (_, example) in zip(axes, example_grid.iterrows()):
    peptide_id = example["peptide_id"]
    drug = example["drug"]

    drug_matrix = per_drug_matrices[drug]
    drug_info = per_drug_designs[drug].copy()

    row = drug_matrix.loc[drug_matrix[PEPTIDE_ID_COL] == peptide_id].iloc[0]
    drug_cols = drug_info["simple_col"].tolist()

    y = pd.to_numeric(row[drug_cols], errors="coerce").to_numpy(dtype=float)
    x = drug_info["log10_concentration"].to_numpy(dtype=float)
    original_conc = drug_info["concentration_nM"].to_numpy(dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)

    result_row = regression_results[
        (regression_results["peptide_id"] == peptide_id)
        & (regression_results["drug"] == drug)
    ].iloc[0]

    slope = result_row["slope"]
    intercept = result_row["intercept"]
    p_value = result_row["p_value"]
    r_squared = result_row["r_squared"]

    x_line = np.linspace(np.nanmin(x[valid]), np.nanmax(x[valid]), 100)
    y_line = intercept + slope * x_line

    labels = [
        "DMSO" if c == 0 else f"{c:g}"
        for c in original_conc
    ]

    ax.scatter(x[valid], y[valid], s=28)
    ax.plot(x_line, y_line, linewidth=1.5)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)

    ax.set_title(
        f"{peptide_id}\n{drug}\n"
        f"slope={slope:.3g}, p={p_value:.2g}, R²={r_squared:.3g}",
        fontsize=9
    )

    ax.set_xlabel("Concentration condition", fontsize=8)
    ax.set_ylabel("log2 abundance", fontsize=8)

fig.suptitle(
    "Example peptide-drug dose-response curves",
    fontsize=14
)

fig.tight_layout(rect=[0, 0, 1, 0.95])

combined_path = FIG_DIR / "fig_response_examples_3x2.png"
fig.savefig(combined_path, dpi=300)
display(fig)
plt.close(fig)

print("Saved:", combined_path)

<Figure size 1500x800 with 6 Axes>

Saved: Approach_1_analysis_outputs\report_figures\fig_response_examples_3x2.png
